In [2]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import os
from typing import Any, Callable, Dict, Optional, Union, Iterable, Tuple

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import sys
ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))
from pathlib import Path
from stats.masking import mask_feature
from stats.filtering import apply_filters, FilterValue

# k-means 군집 분석 수행
#  - 자동선택 함수(silhouette 기준) 포함
#  - 최적의 k를 찾음, best_k와 k별 결과 요약표 반환
# =========================================================
# 1) k 선택 함수: silhouette 점수 기준으로 최적 k 선택
def choose_best_k(X_scaled: np.ndarray, k_range=range(2, 20), random_state=42, min_cluster_size=None) -> tuple[int, pd.DataFrame]:
    rows = []
    best_k = None
    best_score = -1

    # k마다 반복
    for k in k_range:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=30)
        labels = model.fit_predict(X_scaled)

        # Optional size check (skip unstable solutions with tiny clusters)
        if min_cluster_size is not None:
            sizes = np.bincount(labels) # 각 클러스터의 크기 계산
            if sizes.min() < min_cluster_size:
                rows.append({
                    "k": k,
                    "silhouette": np.nan, # skipped, so NaN
                    "inertia": float(model.inertia_), # skipped, but still report inertia
                    "min_cluster_size": int(sizes.min()), # skipped, but still report min size
                    "note": "skipped (tiny cluster)" # skipped due to tiny cluster
                })
                continue
        # Silhouette 점수 계산
        score = silhouette_score(X_scaled, labels)
        # 결과 기록
        rows.append({
            "k": k,
            "silhouette": float(score),
            "inertia": float(model.inertia_),
            "min_cluster_size": int(np.bincount(labels).min()),
            "note": ""
        })
        # 최적 k 갱신
        if score > best_score:
            best_score = score
            best_k = k

    return best_k, pd.DataFrame(rows).sort_values("k").reset_index(drop=True) 

# --------------------------------------------------------
# 2) main: 클러스터링 실행 + 요약 만들기
def run_clustering(X: pd.DataFrame,
                   k: Optional[Union[int, Iterable[int]]] = None,
                   random_state=42, N_INIT=50) -> Tuple[pd.DataFrame, pd.DataFrame, int, Optional[pd.DataFrame]]:
    # 표준화
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    # k 자동선택
    k_table = None
    if k is None or (isinstance(k, (list, tuple)) and len(k) > 1):
        best_k, k_table = choose_best_k(X_scaled, k_range=k, random_state=random_state)
        k = best_k
    
    # 최종 모델
    model = KMeans(n_clusters=k, random_state=random_state, n_init=N_INIT)
    labels = model.fit_predict(X_scaled)

    # 결과 DF
    out = X.copy()
    out.insert(0, "cluster", labels)
    out = out.reset_index()  # <- brings OUTCOME_COL back as a normal column

    # 클러스터 요약: 각 클러스터의 ASP별 평균 feature
    summary = out.groupby("cluster")[X.columns].mean()
    summary.insert(0, "size", out.groupby("cluster").size())

    return out, summary, k, k_table



In [ ]:
from datetime import datetime
    
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
# =========================
# 1) 데이터 경로
# =========================
BASE_DIR = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun")
CSV_PATH = BASE_DIR / "csv" / "통계용" / "2025-12-26_01-39_logodds_V_No_by_ASP_No_ID.csv"

# =========================================================
# 2) 설정값들
# =========================================================
# 컬럼명(파일마다 바뀌면 여기만 바꾸면 됨) 
# 데이터 오브젝트 식별자: n개(OUTCOME_COL)의 d-차원(DIMENTION_COL), 각 (n × d) 조합에 피처 값(FEATURE_COL)
OUTCOME_COL = "outcome_level"
DIMENTION_COL = "ASP_No"
FEATURE_COL = "log_odds"   # or "z"_score", "p_value", etc.

# ✅ Option: 추가할 데이터가 있는 경우
additional_feature = True  # 예: True/False 토글
ADDED_FEATURE_FILE = BASE_DIR / "csv" / "통계용" / "2025-12-27_11-49_logodds_EP_T_by_V_No_ID.csv"
ADDED_OUTCOME_COL = "V_No"
ADDED_FEATURE_COL = "log_odds"
ADDED_FILTERS = {"outcome_level": True}
ADDED_COL_NEW_NAME = "EP_T"  # 추가할 피처명

# 차원 집합: 예) 사용할 상표현(ASP_No)
DIMENTION_SET = [101, 102, 111, 112, 123, 122, 124, 113, 125, 126, 132, 131]

# outcome_level 조건
OUTCOME_MIN = 1000
OUTCOME_EXCLUDE = {2999, 1009, 4999}

# ✅ 필터는 여기 한 번만
FILTERS: Dict[str, FilterValue] = {
    DIMENTION_COL: DIMENTION_SET,
    OUTCOME_COL: lambda s: (s >= OUTCOME_MIN) & (~s.isin(list(OUTCOME_EXCLUDE))),
    "outcome_total": lambda s: s >= 2000,
    # ---- 원하는 필터 있으면 여기 추가 ----
    # "category": ["강의", "낭독"],
    # "outcome_total": lambda s: s >= 500,
}

# 마스킹 옵션(원하면 켜기)
MASK_OPTS = dict(
    p_col="p_value",              # p-value 컬럼명
    p_max=0.05,                    # 예: 0.05, None
    count_col="a_in_context_and_level", # 카운트 컬럼명
    count_min=None,                # 예: 5, 10, None
    abs_min=None,                  # 예: 0.2, 0.5, None
    require_notna=None,            # 예: ["log_odds", "p_value"], None
    fill_value=0.0,                # 마스킹된 값 대체값 (예: 0.0, None  for NaN)
)

# 최소 클러스터 크기: 너무 작은 클러스터는 건너뜀 (선택)
MIN_CLUSTER_SIZE: Optional[int] = None   # 예: 5, 10, None

# outcome_level(행) 단위 추가 필터: 0 아닌 feature가 너무 적으면 제거
MIN_NONZERO_FEATURES: Optional[int] = 3    # 예: 3, 4, None

# k 여러 개를 한 번에 + k마다 여러 번 반복
K_VALUES = list(range(2, 21))   # 2~20
N_REPEATS_PER_K = 30
BASE_RANDOM_STATE = 42

# 출력 파일 경로
OUT_DIR = BASE_DIR / "csv" / "cluster_k_means"
os.makedirs(OUT_DIR, exist_ok=True)

   
# 출력 파일 경로들
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
OUT_FEATURE_MATRIX = OUT_DIR / f"feature_matrix_X_{timestamp}.csv"
OUT_K_TABLE_CSV = OUT_DIR / f"k_selection_table_{timestamp}.csv"
OUT_LABEL_WIDE = OUT_DIR / f"cluster_labels_wide_{timestamp}.csv"
OUT_CLUSTERED = OUT_DIR / f"cluster_outcome_level_k{timestamp}.csv"
OUT_SUMMARY = OUT_DIR / f"cluster_summary_k{timestamp}.csv" 

# =========================================================
# 0) feature matrix 만들기 (필터 + 마스킹 + pivot)
# 데이터프레임을 받아서 feature matrix 반환: np.ndarray
# =========================================================
def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    # (1) 필터 적용
    df2 = apply_filters(df, FILTERS) #stats/filtering.py 참조

    # (2) 마스킹된 feature 만들기
    df2["_feat"] = mask_feature(df2, FEATURE_COL, **MASK_OPTS) #stats/masking.py 참조

    # (3) pivot: outcome_level × ASP_No
    if OUTCOME_COL not in df2.columns:
        raise KeyError(f"Missing OUTCOME_COL: {OUTCOME_COL}")
    if DIMENTION_COL not in df2.columns:
        raise KeyError(f"Missing DIMENTION_COL: {DIMENTION_COL}")

    X = df2.pivot(index=OUTCOME_COL, columns=DIMENTION_COL, values="_feat")

    # (4) DIMENTION 열 고정 + 결측은 0(중립)
    X = X.reindex(columns=DIMENTION_SET).fillna(0.0)

    # (5) (선택) 정보 적은 outcome_level 제거
    if MIN_NONZERO_FEATURES is not None:
        keep = (X != 0).sum(axis=1) >= int(MIN_NONZERO_FEATURES)
        X = X.loc[keep]

    return X

# =========================================================

# =========================================================
#  main
# =========================================================

# 1) 데이터 읽기
print(f"[클러스터링 시작]: {CSV_PATH}, {timestamp} \n")
df = pd.read_csv(CSV_PATH)

# 2) feature matrix 만들기, 필터/마스킹 포함
X = build_feature_matrix(df)
if X.shape[0] < 2:
    raise ValueError(f"필터/전처리 후 {OUTCOME_COL}이 너무 적음. (최소 2개 필요)")
X.to_csv(OUT_FEATURE_MATRIX, encoding="utf-8-sig")     # 참고용: feature matrix 저장
print(f"[feature matrix 생성 완료]: {X.shape}, {OUT_FEATURE_MATRIX}")

# 2-1) 추가 피처 합치기
# optional: 합할 데이터가 있으면 여기에 추가
if additional_feature == True:  # 예: True/False 토글
    
    df_added = pd.read_csv(ADDED_FEATURE_FILE)
    df_added = apply_filters(df_added, ADDED_FILTERS) #stats/filtering.py 참조
    ep_feat = (
        df_added[[ADDED_OUTCOME_COL, ADDED_FEATURE_COL]]              # or your added feature column name
        .drop_duplicates(subset=[ADDED_OUTCOME_COL])
        .set_index(ADDED_OUTCOME_COL)
        .rename(columns={ADDED_FEATURE_COL: ADDED_COL_NEW_NAME, ADDED_OUTCOME_COL: OUTCOME_COL})
    )
X = X.reset_index().set_index(OUTCOME_COL).join(ep_feat, how="left").fillna(0.0)

# 3) clustering
clustered, summary, chosen_k, k_table = run_clustering(
    X, k=K_VALUES, random_state=42, N_INIT=50)

# 4) 결과 저장
if k_table is not None:
    print("\n[클러스터링 k별 요약]")
    print(k_table)
    print(f"\n[선택된 최적 k: {chosen_k}]")
    k_table.to_csv(OUT_K_TABLE_CSV, index=False, encoding="utf-8-sig")
    print(f"[k 선택표 저장 완료: {OUT_K_TABLE_CSV}]")

print("\n[Done]")
print(f"[최종 선택된 k: {chosen_k}]")
print(f"[클러스터링 결과 요약]")
print(summary)


In [ ]:
# 파일로 결과 저장

clustered.to_csv(OUT_CLUSTERED, index=False, encoding="utf-8-sig")
summary.to_csv(OUT_SUMMARY, encoding="utf-8-sig")
print(f"\n[클러스터링 결과 저장 완료: \n{OUT_CLUSTERED}, \n{OUT_SUMMARY}]")
# wide 형식으로 라벨 저장
clustered_wide = clustered.reset_index().pivot(
    index=OUTCOME_COL,
    columns="cluster",
    values="cluster"
)
clustered_wide.columns = [f"cluster_{col}" for col in clustered_wide.columns]
clustered_wide.reset_index(inplace=True)
clustered_wide.to_csv(OUT_LABEL_WIDE, index=False, encoding="utf-8-sig")
print(f"[클러스터 라벨(wide) 저장 완료: {OUT_LABEL_WIDE}]")



In [ ]:
# =========================================================
# 6) 추가 분석: 특정 군집만 다시 군집화
# =========================================================

# 1차 결과 (clustered는 outcome_level + cluster + 12피처)
sub = clustered[clustered["cluster"] == 0].copy()   # 예: 1번 군집만 다시 보기

# 2차 군집 입력 X2: ID를 index로 둠
# 추가한 열이 있는 경우
if additional_feature == True:  
    X2 = sub.set_index(OUTCOME_COL)[DIMENTION_SET+[ADDED_COL_NEW_NAME]]
    
# 추가한 열이 없는 경우
else:   
    X2 = sub.set_index(OUTCOME_COL)[DIMENTION_SET]

# 2차 군집
clustered2, summary2, chosen_k2, k_table2 = run_clustering(
    X2, k=9, random_state=42, N_INIT=100
)

print("\n[2차 군집화 완료]")
if k_table2 is not None:
    print("\n[2차 클러스터링 k별 요약]")
    print(k_table2)
    print(f"\n[2차 선택된 최적 k: {chosen_k2}]") 



[2차 군집화 완료]


In [ ]:
OUT_CLUSTERED = OUT_DIR / f"cluster_outcome_level2_{timestamp}.csv"
OUT_SUMMARY = OUT_DIR / f"cluster_summary2_{timestamp}.csv" 

clustered2.to_csv(OUT_CLUSTERED, index=False, encoding="utf-8-sig")
summary2.to_csv(OUT_SUMMARY, encoding="utf-8-sig")
print(f"\n[클러스터링 결과 저장 완료: \n{OUT_CLUSTERED}, \n{OUT_SUMMARY}]")
print(summary2)


[클러스터링 결과 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_outcome_level2_2025-12-27_12-25.csv, C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_summary2_2025-12-27_12-25.csv]
         size       101       102       111       112       123       122  \
cluster                                                                     
0          45  0.638764 -3.317157  1.252431  1.628615  0.069276 -0.450177   
1          24  0.152193 -2.990369  0.438010 -0.007504  0.174501  0.209818   
2          32 -0.304998  1.743092  0.585265  0.225417 -1.345900  0.049787   
3          59  0.988199 -3.927155  0.762510 -0.825700 -1.327545 -0.648122   
4          22  0.356544 -1.908404 -2.693098 -2.557221 -2.769581 -0.231157   
5          17  0.543876 -2.638357 -0.191047  0.433904  0.120431  0.818207   
6          38  0.168032 -3.090605  0.536664  0.318075  2.215079 -0.057742   
7          32 -0.454251 -0.775056 -0.678813  0.678754 -